In [ ]:
"""
Restaurant Operations Customer Analytics

Loads the five CSV tables into an in-memory SQLite database, then answers
every question from the Local / Dining / Hospitality / Behavior / Review
insight sections using SQL (via pandas.read_sql). A pure-pandas version of
a couple of the queries is included at the bottom for comparison.

Usage:
    python insights_analysis.py
"""

import sqlite3
import pandas as pd

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 10)

DATA_DIR = "."

# ---------------------------------------------------------------------
# 1. Load CSVs into SQLite
# ---------------------------------------------------------------------
conn = sqlite3.connect(":memory:")

tables = {
    "consumers": "consumers.csv",
    "consumer_preferences": "consumer_preferences.csv",
    "ratings": "ratings.csv",
    "restaurants": "restaurants.csv",
    "restaurant_cuisines": "restaurant_cuisines.csv",
}

for table, filename in tables.items():
    df = pd.read_csv(f"{DATA_DIR}/{filename}", encoding="utf-8-sig")
    df.to_sql(table, conn, if_exists="replace", index=False)


def run(title, query):
    print(f"\n=== {title} ===")
    print(pd.read_sql(query, conn).to_string(index=False))


# ---------------------------------------------------------------------
# LOCAL INSIGHTS
# ---------------------------------------------------------------------
run("1. Consumer distribution by city & state", """
    SELECT City, State, COUNT(*) AS num_consumers
    FROM consumers
    GROUP BY City, State
    ORDER BY num_consumers DESC;
""")

run("2. Age distribution by state", """
    SELECT State,
           CASE WHEN Age < 30 THEN 'Under 30'
                WHEN Age <= 60 THEN '30-60'
                ELSE 'Over 60' END AS Age_Group,
           COUNT(*) AS num_consumers
    FROM consumers
    GROUP BY State, Age_Group
    ORDER BY State, num_consumers DESC;
""")

run("3. Smokers vs non-smokers (%) by city", """
    SELECT City,
           ROUND(100.0 * SUM(CASE WHEN Smoker='Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_smokers,
           ROUND(100.0 * SUM(CASE WHEN Smoker='No'  THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_nonsmokers,
           COUNT(*) AS total_consumers
    FROM consumers
    GROUP BY City;
""")

run("4. Parking availability by city", """
    SELECT City, Parking, COUNT(*) AS num_restaurants
    FROM restaurants
    GROUP BY City, Parking
    ORDER BY City, num_restaurants DESC;
""")

# ---------------------------------------------------------------------
# DINING INSIGHTS
# ---------------------------------------------------------------------
run("5. Parking vs price level", """
    SELECT Price, Parking, COUNT(*) AS num_restaurants
    FROM restaurants
    GROUP BY Price, Parking
    ORDER BY Price, num_restaurants DESC;
""")

run("6. Restaurants by state", """
    SELECT State, COUNT(*) AS num_restaurants
    FROM restaurants
    GROUP BY State
    ORDER BY num_restaurants DESC;
""")

run("7. Franchise vs non-franchise ratings", """
    SELECT rs.Franchise,
           CASE r.Overall_Rating WHEN 0 THEN 'Unsatisfactory'
                                  WHEN 1 THEN 'Satisfactory'
                                  ELSE 'Highly Satisfactory' END AS Rating_Category,
           COUNT(*) AS num_ratings
    FROM ratings r
    JOIN restaurants rs ON r.Restaurant_ID = rs.Restaurant_ID
    GROUP BY rs.Franchise, Rating_Category
    ORDER BY rs.Franchise, num_ratings DESC;
""")

run("8. Top preferred cuisines", """
    SELECT Preferred_Cuisine, COUNT(*) AS num_consumers
    FROM consumer_preferences
    GROUP BY Preferred_Cuisine
    ORDER BY num_consumers DESC
    LIMIT 5;
""")

# ---------------------------------------------------------------------
# HOSPITALITY INSIGHTS
# ---------------------------------------------------------------------
run("9. Alcohol service mix (all restaurants combined)", """
    SELECT Alcohol_Service, COUNT(*) AS num_restaurants,
           ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM restaurants), 2) AS pct
    FROM restaurants
    GROUP BY Alcohol_Service
    ORDER BY num_restaurants DESC;
""")

run("10. Transportation methods used by consumers", """
    SELECT Transportation_Method, COUNT(*) AS num_consumers,
           ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM consumers), 1) AS pct
    FROM consumers
    GROUP BY Transportation_Method
    ORDER BY num_consumers DESC;
""")

run("11. Alcohol service vs consumer ratings", """
    SELECT rs.Alcohol_Service,
           CASE r.Overall_Rating WHEN 0 THEN 'Unsatisfactory'
                                  WHEN 1 THEN 'Satisfactory'
                                  ELSE 'Highly Satisfactory' END AS Rating_Category,
           COUNT(*) AS num_ratings
    FROM ratings r
    JOIN restaurants rs ON r.Restaurant_ID = rs.Restaurant_ID
    GROUP BY rs.Alcohol_Service, Rating_Category
    ORDER BY rs.Alcohol_Service, num_ratings DESC;
""")

run("12. Smoking policy distribution", """
    SELECT Smoking_Allowed, COUNT(*) AS num_restaurants,
           ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM restaurants), 2) AS pct
    FROM restaurants
    GROUP BY Smoking_Allowed;
""")

# ---------------------------------------------------------------------
# BEHAVIOR INSIGHTS
# ---------------------------------------------------------------------
run("13. Common occupations by state", """
    SELECT State, Occupation, COUNT(*) AS num_consumers
    FROM consumers
    GROUP BY State, Occupation
    ORDER BY State, num_consumers DESC;
""")

run("14. Drink level distribution by state", """
    SELECT State, Drink_Level, COUNT(*) AS num_consumers,
           ROUND(100.0 * COUNT(*) / (
               SELECT COUNT(*) FROM consumers c2 WHERE c2.State = consumers.State
           ), 1) AS pct_of_state
    FROM consumers
    GROUP BY State, Drink_Level
    ORDER BY State, num_consumers DESC;
""")

run("15. Marital status vs smoking & drinking habits", """
    SELECT Marital_Status, Smoker, Drink_Level, COUNT(*) AS num_consumers
    FROM consumers
    GROUP BY Marital_Status, Smoker, Drink_Level
    ORDER BY Marital_Status, Smoker, num_consumers DESC;
""")

run("16. Occupation vs budget level", """
    SELECT Occupation, Budget, COUNT(*) AS num_consumers
    FROM consumers
    GROUP BY Occupation, Budget
    ORDER BY Occupation, num_consumers DESC;
""")

# ---------------------------------------------------------------------
# REVIEW INSIGHTS — top 5 restaurants
# ---------------------------------------------------------------------
for label, column in [
    ("17. Top 5 restaurants by FOOD rating", "Food_Rating"),
    ("18. Top 5 restaurants by SERVICE rating", "Service_Rating"),
    ("19. Top 5 restaurants by OVERALL rating", "Overall_Rating"),
]:
    run(label, f"""
        SELECT rs.Name,
               SUM(CASE WHEN r.{column} = 2 THEN 1 ELSE 0 END) AS highly_satisfactory,
               SUM(CASE WHEN r.{column} = 1 THEN 1 ELSE 0 END) AS satisfactory,
               SUM(CASE WHEN r.{column} = 0 THEN 1 ELSE 0 END) AS unsatisfactory,
               COUNT(*) AS total_ratings
        FROM ratings r
        JOIN restaurants rs ON r.Restaurant_ID = rs.Restaurant_ID
        GROUP BY rs.Name
        ORDER BY highly_satisfactory DESC
        LIMIT 5;
    """)

# ---------------------------------------------------------------------
# Pure-pandas equivalent example (no SQL) — restaurants by state
# ---------------------------------------------------------------------
print("\n=== Pure-pandas example: restaurants by state ===")
restaurants_df = pd.read_csv(f"{DATA_DIR}/restaurants.csv", encoding="utf-8-sig")
print(
    restaurants_df.groupby("State").size()
    .sort_values(ascending=False)
    .rename("num_restaurants")
    .to_string()
)

conn.close()


=== 1. Consumer distribution by city & state ===
           City           State  num_consumers
San Luis Potosi San Luis Potosi             86
Ciudad Victoria      Tamaulipas             25
     Cuernavaca         Morelos             22
       Jiutepec         Morelos              5

=== 2. Age distribution by state ===
          State Age_Group  num_consumers
        Morelos  Under 30             19
        Morelos     30-60              5
        Morelos   Over 60              3
San Luis Potosi  Under 30             77
San Luis Potosi   Over 60              6
San Luis Potosi     30-60              3
     Tamaulipas  Under 30             25

=== 3. Smokers vs non-smokers (%) by city ===
           City  pct_smokers  pct_nonsmokers  total_consumers
Ciudad Victoria          8.0            88.0               25
     Cuernavaca         22.7            77.3               22
       Jiutepec          0.0           100.0                5
San Luis Potosi         22.1            75.6          

In [ ]:
"""
Restaurant Operations Customer Analytics — PURE PANDAS (no SQL)

Answers every question from the Local / Dining / Hospitality / Behavior /
Review insight sections using only pandas: groupby, merge, value_counts,
pivot_table, etc.

Usage:
    python insights_analysis_pandas.py
"""

import pandas as pd

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 10)

DATA_DIR = "."

# ---------------------------------------------------------------------
# 1. Load CSVs
# ---------------------------------------------------------------------
consumers = pd.read_csv(f"{DATA_DIR}/consumers.csv", encoding="utf-8-sig")
consumer_preferences = pd.read_csv(f"{DATA_DIR}/consumer_preferences.csv", encoding="utf-8-sig")
ratings = pd.read_csv(f"{DATA_DIR}/ratings.csv", encoding="utf-8-sig")
restaurants = pd.read_csv(f"{DATA_DIR}/restaurants.csv", encoding="utf-8-sig")
restaurant_cuisines = pd.read_csv(f"{DATA_DIR}/restaurant_cuisines.csv", encoding="utf-8-sig")


def show(title, result):
    print(f"\n=== {title} ===")
    print(result.to_string())


# ---------------------------------------------------------------------
# LOCAL INSIGHTS
# ---------------------------------------------------------------------

# 1. Consumer distribution by city & state
result = (
    consumers.groupby(["City", "State"])
    .size()
    .rename("num_consumers")
    .sort_values(ascending=False)
)
show("1. Consumer distribution by city & state", result)

# 2. Age distribution by state
age_bins = [0, 29, 60, 200]
age_labels = ["Under 30", "30-60", "Over 60"]
consumers["Age_Group"] = pd.cut(consumers["Age"], bins=age_bins, labels=age_labels)
result = (
    consumers.groupby(["State", "Age_Group"], observed=True)
    .size()
    .rename("num_consumers")
    .sort_values(ascending=False)
    .sort_index(level="State", sort_remaining=False)
)
show("2. Age distribution by state", result)

# 3. Smokers vs non-smokers (%) by city
smoker_pct = (
    consumers.groupby("City")["Smoker"]
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
    .rename("pct")
    .unstack("Smoker")
)
smoker_pct["total_consumers"] = consumers.groupby("City").size()
show("3. Smokers vs non-smokers (%) by city", smoker_pct)

# 4. Parking availability by city
result = (
    restaurants.groupby(["City", "Parking"], dropna=False)
    .size()
    .rename("num_restaurants")
    .sort_values(ascending=False)
    .sort_index(level="City", sort_remaining=False)
)
show("4. Parking availability by city", result)

# ---------------------------------------------------------------------
# DINING INSIGHTS
# ---------------------------------------------------------------------

# 5. Parking vs price level
result = (
    restaurants.groupby(["Price", "Parking"], dropna=False)
    .size()
    .rename("num_restaurants")
    .sort_values(ascending=False)
    .sort_index(level="Price", sort_remaining=False)
)
show("5. Parking vs price level", result)

# 6. Restaurants by state
result = restaurants.groupby("State").size().rename("num_restaurants").sort_values(ascending=False)
show("6. Restaurants by state", result)

# 7. Franchise vs non-franchise ratings
rating_map = {0: "Unsatisfactory", 1: "Satisfactory", 2: "Highly Satisfactory"}
ratings_merged = ratings.merge(restaurants[["Restaurant_ID", "Franchise"]], on="Restaurant_ID")
ratings_merged["Rating_Category"] = ratings_merged["Overall_Rating"].map(rating_map)
result = (
    ratings_merged.groupby(["Franchise", "Rating_Category"])
    .size()
    .rename("num_ratings")
    .sort_values(ascending=False)
    .sort_index(level="Franchise", sort_remaining=False)
)
show("7. Franchise vs non-franchise ratings", result)

# 8. Top preferred cuisines
result = (
    consumer_preferences["Preferred_Cuisine"]
    .value_counts()
    .rename("num_consumers")
    .head(5)
)
show("8. Top preferred cuisines", result)

# ---------------------------------------------------------------------
# HOSPITALITY INSIGHTS
# ---------------------------------------------------------------------

# 9. Alcohol service mix (all restaurants combined)
counts = restaurants["Alcohol_Service"].value_counts(dropna=False)
pct = (counts / len(restaurants) * 100).round(2)
result = pd.DataFrame({"num_restaurants": counts, "pct": pct})
show("9. Alcohol service mix (all restaurants combined)", result)

# 10. Transportation methods used by consumers
counts = consumers["Transportation_Method"].value_counts(dropna=False)
pct = (counts / len(consumers) * 100).round(1)
result = pd.DataFrame({"num_consumers": counts, "pct": pct})
show("10. Transportation methods used by consumers", result)

# 11. Alcohol service vs consumer ratings
ratings_merged2 = ratings.merge(restaurants[["Restaurant_ID", "Alcohol_Service"]], on="Restaurant_ID")
ratings_merged2["Rating_Category"] = ratings_merged2["Overall_Rating"].map(rating_map)
result = (
    ratings_merged2.groupby(["Alcohol_Service", "Rating_Category"], dropna=False)
    .size()
    .rename("num_ratings")
    .sort_values(ascending=False)
    .sort_index(level="Alcohol_Service", sort_remaining=False)
)
show("11. Alcohol service vs consumer ratings", result)

# 12. Smoking policy distribution
counts = restaurants["Smoking_Allowed"].value_counts(dropna=False)
pct = (counts / len(restaurants) * 100).round(2)
result = pd.DataFrame({"num_restaurants": counts, "pct": pct})
show("12. Smoking policy distribution", result)

# ---------------------------------------------------------------------
# BEHAVIOR INSIGHTS
# ---------------------------------------------------------------------

# 13. Common occupations by state
result = (
    consumers.groupby(["State", "Occupation"], dropna=False)
    .size()
    .rename("num_consumers")
    .sort_values(ascending=False)
    .sort_index(level="State", sort_remaining=False)
)
show("13. Common occupations by state", result)

# 14. Drink level distribution by state
counts = consumers.groupby(["State", "Drink_Level"]).size().rename("num_consumers")
state_totals = consumers.groupby("State").size()
pct = counts / counts.index.get_level_values("State").map(state_totals) * 100
result = pd.DataFrame({"num_consumers": counts, "pct_of_state": pct.round(1)})
result = result.sort_values("num_consumers", ascending=False).sort_index(level="State", sort_remaining=False)
show("14. Drink level distribution by state", result)

# 15. Marital status vs smoking & drinking habits
result = (
    consumers.groupby(["Marital_Status", "Smoker", "Drink_Level"], dropna=False)
    .size()
    .rename("num_consumers")
    .sort_values(ascending=False)
    .sort_index(level=["Marital_Status", "Smoker"], sort_remaining=False)
)
show("15. Marital status vs smoking & drinking habits", result)

# 16. Occupation vs budget level
result = (
    consumers.groupby(["Occupation", "Budget"], dropna=False)
    .size()
    .rename("num_consumers")
    .sort_values(ascending=False)
    .sort_index(level="Occupation", sort_remaining=False)
)
show("16. Occupation vs budget level", result)

# ---------------------------------------------------------------------
# REVIEW INSIGHTS — top 5 restaurants
# ---------------------------------------------------------------------
for label, column in [
    ("17. Top 5 restaurants by FOOD rating", "Food_Rating"),
    ("18. Top 5 restaurants by SERVICE rating", "Service_Rating"),
    ("19. Top 5 restaurants by OVERALL rating", "Overall_Rating"),
]:
    merged = ratings.merge(restaurants[["Restaurant_ID", "Name"]], on="Restaurant_ID")
    summary = merged.groupby("Name")[column].value_counts().unstack(fill_value=0)
    summary = summary.rename(columns={0: "unsatisfactory", 1: "satisfactory", 2: "highly_satisfactory"})
    summary["total_ratings"] = summary.sum(axis=1)
    summary = summary.sort_values("highly_satisfactory", ascending=False).head(5)
    show(label, summary[["highly_satisfactory", "satisfactory", "unsatisfactory", "total_ratings"]])


=== 1. Consumer distribution by city & state ===
City             State          
San Luis Potosi  San Luis Potosi    86
Ciudad Victoria  Tamaulipas         25
Cuernavaca       Morelos            22
Jiutepec         Morelos             5

=== 2. Age distribution by state ===
State            Age_Group
Morelos          Under 30     19
                 30-60         5
                 Over 60       3
San Luis Potosi  Under 30     77
                 Over 60       6
                 30-60         3
Tamaulipas       Under 30     25

=== 3. Smokers vs non-smokers (%) by city ===
Smoker              No   Yes  total_consumers
City                                         
Ciudad Victoria   91.7   8.3               25
Cuernavaca        77.3  22.7               22
Jiutepec         100.0   NaN                5
San Luis Potosi   77.4  22.6               86

=== 4. Parking availability by city ===
City             Parking
Ciudad Victoria  NaN        12
                 Yes         6
              